In [ ]:
import io

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload
from oauth2client.service_account import ServiceAccountCredentials

"""Downloads a file
Args:
    real_file_id: ID of the file to download
Returns : IO object with location.

Load pre-authorized user credentials from the environment.
TODO(developer) - See https://developers.google.com/identity
for guides on implementing OAuth2 for the application.
"""
scopes = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive",
]
filename = "/home/tricktx/.service-account/service-account-sou-da-paz.json"
creds = ServiceAccountCredentials.from_json_keyfile_name(
    filename=filename, scopes=scopes
)

try:
    # create drive api client
    service = build("drive", "v3", credentials=creds)

    file_id = "17763cN6OyBnMGvVOJM4fYMBhBHYK2O_l"

    # pylint: disable=maybe-no-member
    request = service.files().get_media(fileId=file_id)
    file = io.BytesIO()
    downloader = MediaIoBaseDownload(file, request)
    done = False
    while done is False:
        status, done = downloader.next_chunk()
        print(f"Download {int(status.progress() * 100)}.")

except HttpError as error:
    print(f"An error occurred: {error}")
    file = None

Download 100.


In [2]:
df = pd.read_excel(file, sheet_name="1-CAC novos registros ", dtype=str)

In [3]:
df = df.rename(
    columns={
        "ano": "ano",
        "semestre_label": "periodo",
        "mais_recente": "consolidado",
        "RM (código)": "id_regiao_militar",
        "RM (nome)": "area_regiao_militar",
        "UF": "sigla_uf",
        "unidade": "unidade",
        "quant": "quantidade",
        "categ informada": "categoria_informada",
        "categ principal": "categoria_principal",
        "macrocategoria 1": "macrocategoria",
    }
)

In [4]:
df = df[
    [
        "ano",
        "periodo",
        "consolidado",
        "id_regiao_militar",
        "area_regiao_militar",
        "sigla_uf",
        "unidade",
        "quantidade",
        "categoria_informada",
        "categoria_principal",
        "macrocategoria",
    ]
]

In [5]:
df.head()

,ano,periodo,consolidado,id_regiao_militar,area_regiao_militar,sigla_uf,unidade,quantidade,categoria_informada,categoria_principal,macrocategoria
0,2023,2023/parcial,não,1,"RJ, ES",NaN,registro,0,caçador,caçador,novos registros particulares
1,2023,2023/parcial,não,2,SP,NaN,registro,0,caçador,caçador,novos registros particulares
2,2023,2023/parcial,não,3,RS,NaN,registro,0,caçador,caçador,novos registros particulares
3,2023,2023/parcial,não,4,Maior parte MG,NaN,registro,0,caçador,caçador,novos registros particulares
4,2023,2023/parcial,não,5,"PR, SC",NaN,registro,0,caçador,caçador,novos registros particulares


In [6]:
cap = [
    "unidade",
    "categoria_informada",
    "categoria_principal",
    "macrocategoria",
]

for x in cap:
    df[x] = df[x].str.capitalize()

In [7]:
df["consolidado"] = (
    df["consolidado"].replace("não", False).replace("sim", True)
)

/tmp/ipykernel_10725/2884829597.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['consolidado'] = df['consolidado'].replace('não', False).replace('sim', True)


In [10]:
df.head()

,ano,periodo,consolidado,id_regiao_militar,area_regiao_militar,sigla_uf,unidade,quantidade,categoria_informada,categoria_principal,macrocategoria
0,2023,2023/parcial,False,1,"RJ, ES",NaN,Registro,0,Caçador,Caçador,Novos registros particulares
1,2023,2023/parcial,False,2,SP,NaN,Registro,0,Caçador,Caçador,Novos registros particulares
2,2023,2023/parcial,False,3,RS,NaN,Registro,0,Caçador,Caçador,Novos registros particulares
3,2023,2023/parcial,False,4,Maior parte MG,NaN,Registro,0,Caçador,Caçador,Novos registros particulares
4,2023,2023/parcial,False,5,"PR, SC",NaN,Registro,0,Caçador,Caçador,Novos registros particulares


In [11]:
df["sigla_uf"].unique()

array([nan, 'AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA',
       'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO',
       'RR', 'RS', 'SC', 'SE', 'SP', 'TO'], dtype=object)

In [43]:
df[df["sigla_uf"] == "TO"]

,ano,periodo,consolidado,id_regiao_militar,area_regiao_militar,sigla_uf,unidade,quantidade,categoria_informada,categoria_principal,macrocategoria
542,2025,2025/parcial,True,NaN,NaN,TO,Registro,420,NaN,NaN,NaN
543,2025,2025/parcial,True,NaN,NaN,TO,Registro,420,NaN,NaN,NaN


In [ ]:
from io import StringIO

import pandas as pd
import requests


def change_columns_name(url_architecture: str) -> dict[str, str]:
    """Essa função recebe como input uma string com link para uma tabela de arquitetura
    e retorna um dicionário com os nomes das colunas originais e os nomes das colunas
    padronizados

    Returns:
        dict: com chaves sendo os nomes originais e valores sendo os nomes padronizados
    """
    # Converte a URL de edição para um link de exportação em formato csv

    rename_columns = []

    url = url_architecture.replace("edit#gid=", "export?format=csv&gid=")
    # Coloca a arquitetura em um dataframe
    df_architecture = pd.read_csv(
        StringIO(requests.get(url, timeout=10).content.decode("utf-8"))
    )

    # Cria um dicionário de nomes de colunas e tipos de dados a partir do dataframe df_architecture
    column_name_dict = dict(
        zip(
            df_architecture["original_name"],
            df_architecture["name"],
            strict=False,
        )
    )

    for x in column_name_dict.items():
        if x[1] != "(deletado)":
            rename_columns.append(x)

    rename_columns = dict(rename_columns)

    orderning_columns = list(rename_columns.values())

    return rename_columns, orderning_columns


df = change_columns_name(
    url_architecture="https://docs.google.com/spreadsheets/d/1D7Kgw0XrSl4Hf__CXDvJkgNbVHaFtWMpMH3XyHNB_wg/edit#gid=0"
)

['ano', 'periodo', 'consolidado', 'id_regiao_metropolitana', 'area_regiao_metropolitana', 'sigla_uf', 'unidade', 'quantidade', 'categoria_informada', 'categoria_principal', 'macrocategoria']


In [59]:
df

{'ano': 'ano',
 'semestre_label': 'periodo',
 'mais_recente': 'consolidado',
 'RM (código)': 'id_regiao_metropolitana',
 'RM (nome)': 'area_regiao_metropolitana',
 'UF': 'sigla_uf',
 'unidade': 'unidade',
 'quant': 'quantidade',
 'categ informada': 'categoria_informada',
 'categ principal': 'categoria_principal',
 'macrocategoria 1': 'macrocategoria'}

In [38]:
rename_columns = []
for x in df.items():
    if x[1] != "(deletado)":
        rename_columns.append(x)

In [42]:
dict(rename_columns)

{'ano': 'ano',
 'semestre_label': 'periodo',
 'referência': 'referencia',
 'UF': 'sigla_uf',
 'RM (código)': 'id_regiao_militar',
 'RM (área)': 'area_regiao_militar',
 'unidade': 'unidade',
 'quant': 'quantidade',
 'macrocategoria 1': 'macrocategoria',
 'microcatgoria 1': '(excluida)'}

In [15]:
df.items()

dict_items([('ano', 'ano'), ('semestre_label', 'periodo'), ('referência', 'referencia'), ('UF', 'sigla_uf'), ('RM (código)', 'id_regiao_militar'), ('RM (área)', 'area_regiao_militar'), ('unidade', 'unidade'), ('quant', 'quantidade'), ('macrocategoria 1', 'macrocategoria'), ('semestre', '(deletado)'), ('mês', '(deletado)'), ('mais_recente', '(deletado)'), ('RM', '(deletado)'), ('categ informada', '(deletado)'), ('categ principal', '(deletado)'), ('região do país', '(deletado)'), ('microcatgoria 1', '(excluida)'), ('microcatgoria 2', '(deletado)'), ('mcrocategoria 3', '(deletado)'), ('microcategoria 4', '(deletado)'), ('microcategoria 5', '(deletado)'), ('microcategoria 6', '(deletado)'), ('macrocategoria 2', '(deletado)'), ('macrocategoria 3', '(deletado)'), ('macrocategoria 4', '(deletado)'), ('macrocategoria 5', '(deletado)'), ('macrocategoria 6', '(deletado)'), ('Documento', '(deletado)'), ('Protocolo', '(deletado)')])

In [3]:
import pandas as pd

lista = ["a", "b", "c"]

assert lista == ["a", "b", "c"]

In [5]:
df = pd.read_csv("/home/tricktx/project_pipeline_anp/cac_acervos_armas.csv")

In [ ]:
import numpy as np

consolidade = set(df["consolidado"].unique())

assert consolidade == {False, True, np.nan}

{False, True, nan}


In [26]:
pd.isna(consolidade)

array([False, False,  True])

In [25]:
import pandas as pd

vals = df["consolidado"].unique()

assert list(vals[~pd.isna(vals)]) == [False, True]

In [47]:
from unidecode import unidecode

total = "não"

unidecode(total)

'nao'

In [20]:
df["consolidado"].unique()

array([False, True, nan], dtype=object)

In [23]:
list(vals[~pd.isna(vals)])

[False, True]

In [22]:
pd.isna(vals).any()

np.True_

In [1]:
import pandas as pd

df = pd.read_csv("/home/tricktx/project_pipeline_anp/cac_pessoas.csv")

In [4]:
df.columns = df.columns.str.strip()

In [44]:
for x in df.columns:
    if x in [
        "unidade",
        "categoria_informada",
        "categoria_principal",
        "macrocategoria",
    ]:
        print(x)
        df[x].str.capitalize()

unidade
macrocategoria


In [23]:
from sheets.constants import constants

constants.tabelas.value["cac_novos_registros"]

{'real_file_id': '17763cN6OyBnMGvVOJM4fYMBhBHYK2O_l',
 'sheet_name': '1-CAC novos registros ',
 'url_architecture': 'https://docs.google.com/spreadsheets/d/1D7Kgw0XrSl4Hf__CXDvJkgNbVHaFtWMpMH3XyHNB_wg/edit#gid=0'}

In [26]:
from constants import constants

constants.tabelas.value["cac_novos_registros"]

{'name_table': 'cac_novos_registros',
 'sheet_name': '1-CAC novos registros ',
 'url_architecture': 'https://docs.google.com/spreadsheets/d/1D7Kgw0XrSl4Hf__CXDvJkgNbVHaFtWMpMH3XyHNB_wg/edit#gid=0'}